# EMFF Ratio Construction & Plot

This notebook constructs the EMFF ratio $R^{fi}(\tau,t_s)$ from HDF5 3pt and 2pt data using Eq. (6) of arXiv:2102.06047.
It saves the ratio tables and plots the results.

No fitting is performed — this is a pure data-extraction + visualisation workflow.


## Imports / Setup


In [ ]:
from pathlib import Path
import sys

# --- Set REPO_ROOT to the lat-hadron-analysis repository ---
# Try: 1) current directory  2) environment variable  3) hard-coded path
REPO_ROOT = None
for _candidate in [
    Path.cwd().resolve(),
    Path(__file__).resolve().parent if '__file__' in dir() else None,
    Path('/Users/xiang/Desktop/codes/lat-hadron-analysis'),  # <-- EDIT THIS
]:
    if _candidate is not None and (_candidate / 'src' / 'lqcd_analysis').is_dir():
        REPO_ROOT = _candidate
        break
if REPO_ROOT is None:
    raise FileNotFoundError(
        'Cannot find lat-hadron-analysis repository. '
        'Edit the hard-coded path in the cell above.'
    )
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print(f'REPO_ROOT: {REPO_ROOT}')

import numpy as np
import matplotlib.pyplot as plt

from lqcd_analysis.emff.io import (
    resolve_emff_h5_path,
    load_emff_correlator,
    load_emff_c2pt_correlator,
    expand_template,
)
from lqcd_analysis.common.bootstrap import (
    bin_samples,
    bootstrap_indices,
    bootstrap_means,
)
from lqcd_analysis.common.utils import robust_mean_and_error
from lqcd_analysis.emff.models import compute_tau_range_for_tsep
from lqcd_analysis.emff.fit_nstate import _compute_emff_ratio, _hadron_energy_from_dispersion

print('Imports OK')


## User Inputs

Edit the paths and parameters below for your analysis.


In [ ]:
# --- Paths ---
DATA_3PT = Path("/Users/xiang/Desktop/docs/0-2026/projects/3-EMFF/data/l48c64a060_m140/pion_EMFF")
DATA_2PT = Path("/Users/xiang/Desktop/docs/0-2026/projects/3-EMFF/data/l48c64a060_m140/c2pt")
OUTPUT_DIR = Path("results_ratio_transverse_avg")

# --- Operators ---
SRC_GAMMA = "5"       # source gamma
SNK_GAMMA = "5"       # sink gamma under SS/ in the 2pt HDF5 file
INS_GAMMA = "T"       # current insertion gamma (T = gamma_t)

# --- Lattice and hadron settings ---
NS = 48
LATTICE_SPACING_FM = 0.060
HADRON_MASS_GEV = 0.140

# --- Momentum ---
PFX, PFY, PFZ = 0, 0, 0   # final momentum Pf
QXLIST = [-2, -1, 0, 1, 2]
QYLIST = [-2, -1, 0, 1, 2]
QZLIST = [-2, -1, 0, 1, 2]
AVERAGE_TRANSVERSE_ORBITS = True

# --- Time separations ---
TSLIST = [4, 6, 8, 10, 12]

# --- Tau window (skip endpoints near source/sink) ---
# tau_min: minimum tau (inclusive).  tau_offset: from tsep; -1 = tsep-1
TAU_MIN = 1
TAU_OFFSET = -1  # -1 means skip tau=0 and tau=tsep

# --- Bootstrap ---
BINSIZE = 1
BOOTSTRAP_SAMPLES = None   # None = auto (use n_cfg)
BOOTSTRAP_SIZE = None      # None = auto
SEED = 2026

# --- Plot options ---
PLOT_REAL = True
PLOT_IMAG = True

# --- 3pt HDF5 path and dataset templates ---
C3PT_PATH = str(
    DATA_3PT / "l48c64a060.pion_EMFF.CFG.EMFF.ex.SRC.1HYP_M140_GSRC_W52_5."
    "posSrc000_posSink000_negSrc000_negSink000.src{src_gamma}."
    "PX{pfx}PY{pfy}PZ{pfz}dt{tsep}.h5"
)
C3PT_DATASET_PATH = "SS/{insert_gamma}/PX{qx}PY{qy}PZ{qz}"

# --- 2pt HDF5 path template ---
C2PT_PATH = str(
    DATA_2PT / "l48c64a060.c2pt.CFG.EMFF.ex.SRC.1HYP_M140_GSRC_W52_5."
    "posSrc000_posSink000_negSrc000_negSink000.src{src_gamma}.h5"
)

print(f"Output dir: {OUTPUT_DIR.resolve()}")
print(f"q combinations before orbit averaging: {len(QXLIST)}x{len(QYLIST)}x{len(QZLIST)} = {len(QXLIST)*len(QYLIST)*len(QZLIST)}")
print(f"tsep values: {TSLIST}")


## Load 2pt Correlator


In [ ]:
# Load final-momentum 2pt once. Initial-momentum 2pt is loaded per q below.
pz_2pt = int(round(np.sqrt(PFX**2 + PFY**2 + PFZ**2)))

c2pt_path = expand_template(
    C2PT_PATH,
    src_gamma=SRC_GAMMA,
    sink_gamma=SNK_GAMMA,
    pz=pz_2pt,
    pfx=PFX,
    pfy=PFY,
    pfz=PFZ,
)
print(f"Loading final-state 2pt from: {c2pt_path}")

times_2pt, c2pt_final = load_emff_c2pt_correlator(
    c2pt_path,
    sink_gamma=SNK_GAMMA,
    px=PFX,
    py=PFY,
    pz=PFZ,
)
energy_final = _hadron_energy_from_dispersion(
    (PFX, PFY, PFZ),
    ns=NS,
    lattice_spacing_fm=LATTICE_SPACING_FM,
    hadron_mass_gev=HADRON_MASS_GEV,
)
print(f"2pt shape: {c2pt_final.shape} (n_cfg={c2pt_final.shape[0]}, Nt={c2pt_final.shape[1]})")
print(f"E_final = {energy_final:.6f} GeV")


## Construct Ratio for All (q, tsep)


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tables_dir = OUTPUT_DIR / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)

if AVERAGE_TRANSVERSE_ORBITS and (PFX != 0 or PFY != 0):
    raise ValueError("transverse orbit averaging currently assumes P_f has no transverse momentum")

qx_set = set(QXLIST)
qy_set = set(QYLIST)


def q_tag_from(qx: int, qy: int, qz: int) -> str:
    return f"q{qx:+d}_{qy:+d}_{qz:+d}".replace("+", "p").replace("-", "m")


def transverse_orbit(qx: int, qy: int, qz: int) -> tuple[tuple[int, int, int], ...]:
    if not AVERAGE_TRANSVERSE_ORBITS:
        return ((qx, qy, qz),)
    members: set[tuple[int, int, int]] = set()
    for ax, ay in {(abs(qx), abs(qy)), (abs(qy), abs(qx))}:
        for sx in ({-1, 1} if ax else {1}):
            for sy in ({-1, 1} if ay else {1}):
                mx, my = sx * ax, sy * ay
                if mx in qx_set and my in qy_set:
                    members.add((mx, my, qz))
    return tuple(sorted(members))


canonical_groups: dict[tuple[int, int, int], tuple[tuple[int, int, int], ...]] = {}
seen_members: set[tuple[int, int, int]] = set()
for qz in QZLIST:
    transverse_pairs = sorted(
        {(max(abs(qx), abs(qy)), min(abs(qx), abs(qy))) for qx in QXLIST for qy in QYLIST}
    )
    for qx_abs, qy_abs in transverse_pairs:
        canonical = (qx_abs, qy_abs, qz)
        members = transverse_orbit(qx_abs, qy_abs, qz)
        if not members:
            continue
        if any(member in seen_members for member in members):
            continue
        canonical_groups[canonical] = members
        seen_members.update(members)

print(f"canonical q groups after transverse orbit averaging: {len(canonical_groups)}")

c2pt_initial_cache: dict[tuple[int, int, int], np.ndarray] = {}

def load_initial_c2pt_for_q(qx: int, qy: int, qz: int) -> np.ndarray:
    pix, piy, piz = PFX - qx, PFY - qy, PFZ - qz
    momentum = (pix, piy, piz)
    if momentum in c2pt_initial_cache:
        return c2pt_initial_cache[momentum]
    pi_label = int(round(np.sqrt(pix**2 + piy**2 + piz**2)))
    c2pt_initial_path = expand_template(
        C2PT_PATH,
        src_gamma=SRC_GAMMA,
        sink_gamma=SNK_GAMMA,
        pz=pi_label,
        pfx=pix,
        pfy=piy,
        pfz=piz,
    )
    _, c2pt_initial = load_emff_c2pt_correlator(
        c2pt_initial_path,
        sink_gamma=SNK_GAMMA,
        px=pix,
        py=piy,
        pz=piz,
    )
    c2pt_initial_cache[momentum] = c2pt_initial
    return c2pt_initial


def average_initial_c2pt(members: tuple[tuple[int, int, int], ...]) -> np.ndarray:
    return np.mean(
        np.stack([load_initial_c2pt_for_q(qx, qy, qz) for qx, qy, qz in members], axis=0),
        axis=0,
    )


def average_c3pt_tau(members: tuple[tuple[int, int, int], ...], tsep: int) -> np.ndarray:
    c3pt_blocks = []
    for qx, qy, qz in members:
        h5_path = resolve_emff_h5_path(
            C3PT_PATH,
            src_gamma=SRC_GAMMA, pfx=PFX, pfy=PFY, pfz=PFZ, tsep=tsep,
        )
        c3pt = load_emff_correlator(
            h5_path, C3PT_DATASET_PATH,
            insert_gamma=INS_GAMMA, qx=qx, qy=qy, qz=qz,
        )
        c3pt_blocks.append(c3pt[:tsep+1, :])
    return np.mean(np.stack(c3pt_blocks, axis=0), axis=0)


# Store all ratio data for later plotting
all_ratio_data: dict[tuple, dict[int, np.ndarray]] = {}  # q -> tsep -> boot_samples (n_boot, tau_count)
orbit_members_by_q: dict[tuple, tuple[tuple[int, int, int], ...]] = {}

for (qx, qy, qz), members in canonical_groups.items():
    q_tag = q_tag_from(qx, qy, qz)
    print(f"\n--- Processing canonical q=({qx:+d},{qy:+d},{qz:+d}) from {len(members)} orbit members ---")
    for member in members:
        print(f"    member q=({member[0]:+d},{member[1]:+d},{member[2]:+d})")

    c2pt_initial = average_initial_c2pt(members)
    pix, piy, piz = PFX - qx, PFY - qy, PFZ - qz
    energy_initial = _hadron_energy_from_dispersion(
        (pix, piy, piz),
        ns=NS,
        lattice_spacing_fm=LATTICE_SPACING_FM,
        hadron_mass_gev=HADRON_MASS_GEV,
    )
    
    ratio_by_tsep: dict[int, np.ndarray] = {}  # tsep -> (n_cfg, tau_count)
    
    for tsep in TSLIST:
        c3pt_tau = average_c3pt_tau(members, tsep)
        ratio = _compute_emff_ratio(
            c3pt_tau,
            c2pt_initial,
            c2pt_final,
            tsep=tsep,
            energy_initial=energy_initial,
            energy_final=energy_final,
        )
        ratio_by_tsep[tsep] = ratio
        print(f"  tsep={tsep}: averaged C3pt shape={c3pt_tau.shape}, ratio shape={ratio.shape}")
    
    # --- Bin ---
    if BINSIZE > 1:
        for tsep in TSLIST:
            ratio_by_tsep[tsep] = bin_samples(ratio_by_tsep[tsep], binsize=BINSIZE)
    
    # --- Bootstrap (same indices across all tsep) ---
    n_cfg = next(iter(ratio_by_tsep.values())).shape[0]
    n_boot = n_cfg if BOOTSTRAP_SAMPLES is None else BOOTSTRAP_SAMPLES
    draw_size = n_cfg if BOOTSTRAP_SIZE is None else BOOTSTRAP_SIZE
    indices = bootstrap_indices(n_cfg, draw_size, seed=SEED + abs(qx*100 + qy*10 + qz), n_boot=n_boot)
    
    boot_ratio_by_tsep: dict[int, np.ndarray] = {}
    for tsep in TSLIST:
        boot_ratio_by_tsep[tsep] = bootstrap_means(
            ratio_by_tsep[tsep], indices=indices
        )  # (n_boot, tsep+1)
    
    all_ratio_data[(qx, qy, qz)] = boot_ratio_by_tsep
    orbit_members_by_q[(qx, qy, qz)] = members
    
    # --- Write ratio table ---
    table_path = tables_dir / f"ratio_src{SRC_GAMMA}_snk{SNK_GAMMA}_ins{INS_GAMMA}_{q_tag}.txt"
    with table_path.open("w", encoding="utf-8") as f:
        f.write(f"canonical_qx {qx} canonical_qy {qy} canonical_qz {qz}\n")
        f.write("averaged_q_members " + " ".join(f"({mx},{my},{mz})" for mx, my, mz in members) + "\n")
        f.write("\t".join([
            "tsep", "tau", "tau_mid",
            "ratio_real_mean", "ratio_real_err",
            "ratio_imag_mean", "ratio_imag_err",
        ]) + "\n")
        for tsep in sorted(boot_ratio_by_tsep.keys()):
            boot = boot_ratio_by_tsep[tsep]  # (n_boot, tsep+1)
            tau_vals = compute_tau_range_for_tsep(tsep, TAU_MIN, TAU_OFFSET)
            for tau in tau_vals:
                real_samples = np.real(boot[:, tau])
                imag_samples = np.imag(boot[:, tau])
                real_mean, real_err = robust_mean_and_error(real_samples)
                imag_mean, imag_err = robust_mean_and_error(imag_samples)
                tau_mid = tau - tsep / 2.0
                f.write("\t".join([
                    str(tsep), str(tau), f"{tau_mid:.1f}",
                    f"{real_mean:.10e}", f"{real_err:.10e}",
                    f"{imag_mean:.10e}", f"{imag_err:.10e}",
                ]) + "\n")
    print(f"  Saved: {table_path}")

print(f"\nDone. Processed {len(all_ratio_data)} canonical q-combinations from {len(seen_members)} raw q-combinations.")


## Plot: Ratio vs $\tau - t_{sep}/2$

All tsep values overlaid for each q. The x-axis is centred at the midpoint of the 3pt function.


In [ ]:
def plot_ratio_for_q(qx, qy, qz, boot_ratio_by_tsep, output_dir):
    """Plot ratio vs tau - tsep/2 for all tsep values, real and imag panels."""
    q_tag = f"q{qx:+d}_{qy:+d}_{qz:+d}".replace("+", "p").replace("-", "m")
    
    # Q^2 in lattice units: (2*pi/Ns)^2 * (qx^2 + qy^2 + qz^2)
    # (approximate, assuming Ns=48)
    q2_lat = (2.0 * np.pi / 48.0)**2 * (qx**2 + qy**2 + qz**2)
    
    tsep_list = sorted(boot_ratio_by_tsep.keys())
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(tsep_list)))
    
    if PLOT_REAL and PLOT_IMAG:
        fig, (ax_r, ax_i) = plt.subplots(1, 2, figsize=(14, 5))
    elif PLOT_REAL:
        fig, ax_r = plt.subplots(1, 1, figsize=(7, 5))
        ax_i = None
    elif PLOT_IMAG:
        fig, ax_i = plt.subplots(1, 1, figsize=(7, 5))
        ax_r = None
    else:
        return
    
    for idx, tsep in enumerate(tsep_list):
        boot = boot_ratio_by_tsep[tsep]  # (n_boot, tsep+1)
        tau_vals = compute_tau_range_for_tsep(tsep, TAU_MIN, TAU_OFFSET)
        x_mid = tau_vals - tsep / 2.0
        color = colors[idx]
        
        real_samples = np.real(boot)
        imag_samples = np.imag(boot)
        
        if ax_r is not None:
            real_mean = np.array([robust_mean_and_error(real_samples[:, t])[0] for t in tau_vals])
            real_err = np.array([robust_mean_and_error(real_samples[:, t])[1] for t in tau_vals])
            ax_r.errorbar(x_mid, real_mean, yerr=real_err,
                         fmt="o-", ms=4, color=color, label=f"tsep={tsep}")
        
        if ax_i is not None:
            imag_mean = np.array([robust_mean_and_error(imag_samples[:, t])[0] for t in tau_vals])
            imag_err = np.array([robust_mean_and_error(imag_samples[:, t])[1] for t in tau_vals])
            ax_i.errorbar(x_mid, imag_mean, yerr=imag_err,
                         fmt="o-", ms=4, color=color, label=f"tsep={tsep}")
    
    if ax_r is not None:
        ax_r.set_xlabel(r"$\tau - t_{sep}/2$")
        ax_r.set_ylabel(r"Re $R(t_{sep}, \tau)$")
        ax_r.axvline(0, color="gray", ls="--", alpha=0.3)
        ax_r.legend(fontsize=7, ncol=2)
        ax_r.set_title(f"Real part  |  q=({qx:+d},{qy:+d},{qz:+d})  |  $Q^2$={q2_lat:.3f}")
    
    if ax_i is not None:
        ax_i.set_xlabel(r"$\tau - t_{sep}/2$")
        ax_i.set_ylabel(r"Im $R(t_{sep}, \tau)$")
        ax_i.axvline(0, color="gray", ls="--", alpha=0.3)
        ax_i.legend(fontsize=7, ncol=2)
        ax_i.set_title(f"Imag part  |  q=({qx:+d},{qy:+d},{qz:+d})  |  $Q^2$={q2_lat:.3f}")
    
    fig.suptitle(f"EMFF Ratio  |  q=({qx:+d},{qy:+d},{qz:+d})", fontsize=13, y=1.02)
    fig.tight_layout()
    
    plot_path = output_dir / f"ratio_src{SRC_GAMMA}_snk{SNK_GAMMA}_ins{INS_GAMMA}_{q_tag}.pdf"
    fig.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Plot saved: {plot_path}")


plots_dir = OUTPUT_DIR / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

for (qx, qy, qz), boot_ratio_by_tsep in all_ratio_data.items():
    plot_ratio_for_q(qx, qy, qz, boot_ratio_by_tsep, plots_dir)

print(f"\nAll plots saved to {plots_dir.resolve()}")


## Option Guide

### Paths
- `DATA_3PT`: Directory containing HDF5 3pt files.
- `DATA_2PT`: Directory containing HDF5 2pt files.
- `OUTPUT_DIR`: Where to write orbit-averaged ratio tables and plots.

### Operators
- `SRC_GAMMA`: Source gamma matrix. Must match the HDF5 file name.
- `SNK_GAMMA`: Sink gamma matrix. Must match the `SS/<sink_gamma>/` group in the 2pt HDF5 file.
- `INS_GAMMA`: Current insertion gamma. For pion EMFF vector current use `"T"` (γ_t).

### Momentum
- `PFX, PFY, PFZ`: Final momentum $P_f$. The 2pt final-state correlator is read from `PX{PFX}PY{PFY}PZ{PFZ}`.
- The plotted ratio uses Eq. (6) of arXiv:2102.06047 with `P_i = P_f - q` and `E(P)=sqrt(hadron_mass_gev^2 + P^2)`.
- `AVERAGE_TRANSVERSE_ORBITS`: When true, average degenerate transverse sign flips and x/y exchanges, and save only canonical non-negative representatives.
- `QXLIST, QYLIST, QZLIST`: Momentum transfer $q$ values to analyse.
- Total q combinations = len(QXLIST) x len(QYLIST) x len(QZLIST).


### Ratio Formula
The workflow builds the ratio using Eq. (6) of arXiv:2102.06047:

```text
R^{fi}(tau, ts)
= [2 sqrt(Ef Ei) / (Ef + Ei)]
  * C3pt(Pf, Pi, tau, ts) / C2pt(ts, Pi)
  * sqrt[
      C2pt(ts - tau, Pf) C2pt(tau, Pi) C2pt(ts, Pi)
      /
      C2pt(ts - tau, Pi) C2pt(tau, Pf) C2pt(ts, Pf)
    ]
```

Here `q = Pf - Pi`, so `Pi = Pf - q`. Energies use the input hadron mass:

```text
E(P) = sqrt(hadron_mass_gev^2 + |P|^2),
P = 2 pi n / (a Ns).
```

When transverse orbit averaging is enabled, the workflow first averages the degenerate transverse orbit members at the correlator level (`C3pt` and the corresponding initial-state `C2pt`) and then applies the ratio formula to the averaged correlators. It does not average `qz` with `-qz`.

### Time Separation
- `TSLIST`: Time separation values. Each must have a corresponding HDF5 file.
- `TAU_MIN, TAU_OFFSET`: Tau window for each tsep. `TAU_OFFSET = -1` means tau goes up to tsep-1. Set both to 0 to include all tau values.

### Bootstrap
- `BINSIZE`: Bin size before bootstrap (1 = no binning).
- `BOOTSTRAP_SAMPLES`: Number of bootstrap samples (`None` = auto = n_cfg).
- `SEED`: Random seed for reproducibility.

### Output
- Ratio tables: `tables/ratio_q{+qx}_{+qy}_{+qz}.txt`
  - Columns: tsep, tau, tau_mid, ratio_real_mean, ratio_real_err, ratio_imag_mean, ratio_imag_err
- Ratio plots: `plots/ratio_q{+qx}_{+qy}_{+qz}.pdf`
  - Real and imag panels, all tsep overlaid, x-axis = tau - tsep/2
